In [46]:
from pathlib import Path
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

from constants import (
    UCI_CLIENT_SITES_TO_VALIDATE,
    UCI_DATA_FREQUENCY_MINUTES,
    UCI_VALIDATION_START,
    UCI_VALIDATION_WINDOW,
)

In [48]:
def mean_absolute_error(y_true: np.ndarray, y_hat: np.ndarray) -> float:
    return np.mean(np.abs(y_true - y_hat))


def weighted_absolute_pct_error(y_true: np.ndarray, y_hat: np.ndarray) -> float:
    return np.sum(np.abs(y_true - y_hat)) / np.sum(np.abs(y_true))


def mean_squared_scaled_error(y_true: np.ndarray, y_hat: np.ndarray, y_train: np.ndarray, period: int = 1) -> float:
    # Calculate the in sample naive train error
    y_train_idx = np.arange(len(y_train))
    y_hat_naive = y_train[y_train_idx - period]
    naive_errors = (y_train - y_hat_naive)[period:]
    mse_naive = np.mean(naive_errors ** 2)
    
    # Calculate forecast errors
    mse_forecast = np.mean((y_true - y_hat) ** 2)
    
    return  mse_forecast / mse_naive

In [36]:
# Constants

UCI_DATA_DIR = Path("../../data/uci")
UCI_DATA_FILE_NAME = "preprocessed.pq"
UCI_DATA_PATH = UCI_DATA_DIR / UCI_DATA_FILE_NAME

N_VALIDATION_FOLDS = 10
SEASONAL_PERIOD = 96

MODELS = ["naive", "ets", "tcn"]

RESULTS_OUTPUT_DIR = Path("../../results/uci")

In [37]:
# Load preprocessed data

uci_df = pl.read_parquet(UCI_DATA_PATH)
UCI_CLIENT_DF = uci_df.filter(pl.col("client").is_in(UCI_CLIENT_SITES_TO_VALIDATE))

In [53]:
site_model_errors = {}

for site in UCI_CLIENT_SITES_TO_VALIDATE:
    client_df = UCI_CLIENT_DF.filter(pl.col("client") == site)

    model_errors = defaultdict(list)
    for fold in range(N_VALIDATION_FOLDS):
        val_start = UCI_VALIDATION_START + fold * UCI_VALIDATION_WINDOW
        val_end = val_start + UCI_VALIDATION_WINDOW
        train_df = (
            client_df
            .filter(pl.col("timestamp").lt(val_start))
            .select(pl.col("timestamp"), pl.col("demand"))
            .sort(by=pl.col("timestamp"))
        )
        y_train = train_df["demand"].to_numpy()
    
        for model in MODELS:
            model_site_dir = RESULTS_OUTPUT_DIR / model / site
            forecasts_file = model_site_dir / f"forecasts_{site}_fold_{fold}.pq"
            
            forecasts_df = pl.read_parquet(forecasts_file)
            y_true = forecasts_df["demand"].to_numpy()
            y_hat = forecasts_df["forecast"].to_numpy()
            
            mae = mean_absolute_error(y_true, y_hat),
            wape = weighted_absolute_pct_error(y_true, y_hat)
            rmsse = np.sqrt(mean_squared_scaled_error(y_true, y_hat, y_train, period=SEASONAL_PERIOD))
            model_error_dict = {
                "fold_index": fold,
                "mean_absolute_error": mae,
                "weighted_absolute_pct_error": wape,
                "root_mean_squared_scaled_error": rmsse
            }
            model_errors[model].append(model_error_dict)

    site_model_errors[site] = model_errors

In [ ]:
# Visualise errors ...
site_model_errors

{'MT_156': defaultdict(list,
             {'naive': [{'fold_index': 0,
                'mean_absolute_error': (np.float64(42.13967300894161),),
                'weighted_absolute_pct_error': np.float64(0.47483859724401656),
                'root_mean_squared_scaled_error': np.float64(1.649777048581183)},
               {'fold_index': 1,
                'mean_absolute_error': (np.float64(10.080773029735921),),
                'weighted_absolute_pct_error': np.float64(0.11050578169532294),
                'root_mean_squared_scaled_error': np.float64(0.5224894515009355)},
               {'fold_index': 2,
                'mean_absolute_error': (np.float64(18.368131368267843),),
                'weighted_absolute_pct_error': np.float64(0.2396734017625522),
                'root_mean_squared_scaled_error': np.float64(0.8602044058571423)},
               {'fold_index': 3,
                'mean_absolute_error': (np.float64(26.395625389893947),),
                'weighted_absolute_pct_error': n